# O que tem nos arquivos baixados

Dois arquivos já estão no cache do PySUS. Os dois são do **Acre**, competência **janeiro de 2024**.

| Arquivo | Sistema | O que é uma linha |
| --- | --- | --- |
| `RDAC2401.parquet` | SIH, AIH reduzida | uma internação hospitalar |
| `PAAC2401.parquet` | SIA, produção ambulatorial | um procedimento ambulatorial aprovado (a quantidade vai em `PA_QTDAPR`) |

As colunas chegam como texto, do jeito que o DATASUS grava. Abaixo ficam só as colunas que ajudam a ler um registro. A lista completa aparece em cada seção.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 24)

cache = Path.home() / "pysus" / "downloads" / "ducklake"
arquivo_sih = cache / "sih" / "RDAC2401.parquet"
arquivo_sia = cache / "sia" / "PAAC2401.parquet"

for caminho in (arquivo_sih, arquivo_sia):
    tamanho_mb = caminho.stat().st_size / 1024 / 1024
    print(f"{caminho.name:20}  {tamanho_mb:6.2f} MB  {caminho}")

## Internações hospitalares (SIH)

Cada linha é uma AIH (autorização de internação hospitalar) paga pelo SUS.

| Coluna | Significado |
| --- | --- |
| `N_AIH` | número da AIH |
| `CNES` | estabelecimento onde ocorreu a internação |
| `MUNIC_RES` | município de residência do paciente (código IBGE) |
| `SEXO` | `1` masculino, `3` feminino |
| `COD_IDADE` + `IDADE` | unidade e número da idade. `2` = dias, `3` = meses, `4` = anos, `5` = 100 anos mais o valor de `IDADE` |
| `RACA_COR` | `01` branca, `02` preta, `03` parda, `04` amarela, `05` indígena |
| `DT_INTER`, `DT_SAIDA` | entrada e saída, no formato `AAAAMMDD` |
| `DIAS_PERM` | dias de permanência |
| `DIAG_PRINC` | diagnóstico principal (CID-10) |
| `PROC_REA` | procedimento realizado |
| `VAL_TOT` | valor total pago pelo SUS |
| `MORTE` | `1` se houve óbito na internação, `0` caso contrário |
| `ANO_CMPT`, `MES_CMPT` | ano e mês da competência |

In [ ]:
sih = pd.read_parquet(arquivo_sih)
print(f"{len(sih):,} internações e {sih.shape[1]} colunas")
print(sih.columns.tolist())

In [ ]:
colunas_sih = [
    "N_AIH", "CNES", "MUNIC_RES", "SEXO", "COD_IDADE", "IDADE", "RACA_COR",
    "DT_INTER", "DT_SAIDA", "DIAS_PERM", "DIAG_PRINC", "PROC_REA",
    "VAL_TOT", "MORTE", "ANO_CMPT", "MES_CMPT",
]

sih[colunas_sih].head(12)

In [ ]:
print("Uma internação, coluna por coluna:")
sih.loc[0, colunas_sih].to_frame("valor")

In [ ]:
print("Sexo")
print(sih["SEXO"].value_counts())
print("\nÓbito na internação")
print(sih["MORTE"].value_counts())
print("\nUnidade da idade (2=dias, 3=meses, 4=anos, 5=100+anos)")
print(sih["COD_IDADE"].value_counts())
print("\nDiagnósticos principais mais frequentes")
print(sih["DIAG_PRINC"].value_counts().head(10))

## Atendimentos ambulatoriais (SIA)

Cada linha é produção ambulatorial aprovada, não necessariamente uma pessoa. O campo `PA_QTDAPR` diz quantos procedimentos aquela linha representa. Quando `PA_SEXO` é `0`, `PA_IDADE` é `999` e `PA_CIDPRI` é `0000`, a linha está consolidada: o DATASUS somou vários atendimentos iguais e não guardou o paciente.

| Coluna | Significado |
| --- | --- |
| `PA_CODUNI` | estabelecimento (CNES) |
| `PA_UFMUN` | município do estabelecimento |
| `PA_CMP` | competência do procedimento, `AAAAMM` |
| `PA_PROC_ID` | procedimento da tabela SUS |
| `PA_CIDPRI` | CID principal. `0000` = não informado |
| `PA_IDADE` | idade em anos. `999` = não se aplica (linha consolidada) |
| `PA_SEXO` | `M`, `F` ou `0` (consolidado) |
| `PA_RACACOR` | mesma escala do SIH; `00` = sem informação |
| `PA_QTDAPR` | quantidade aprovada |
| `PA_VALAPR` | valor aprovado |

In [ ]:
sia = pd.read_parquet(arquivo_sia)
print(f"{len(sia):,} linhas e {sia.shape[1]} colunas")
print(sia.columns.tolist())

In [ ]:
colunas_sia = [
    "PA_CODUNI", "PA_UFMUN", "PA_CMP", "PA_PROC_ID", "PA_CIDPRI",
    "PA_IDADE", "PA_SEXO", "PA_RACACOR", "PA_QTDAPR", "PA_VALAPR",
]

sia[colunas_sia].head(12)

In [ ]:
print("Uma linha ambulatorial, coluna por coluna:")
sia.loc[0, colunas_sia].to_frame("valor")

In [ ]:
print("Sexo (0 = linha consolidada, sem paciente)")
print(sia["PA_SEXO"].value_counts())
print("\nProcedimentos mais frequentes")
print(sia["PA_PROC_ID"].value_counts().head(10))
print("\nQuantidade aprovada: total e algumas linhas")
print("soma de PA_QTDAPR =", pd.to_numeric(sia["PA_QTDAPR"]).sum())
print(sia["PA_QTDAPR"].value_counts().head(8))